# SRMatcher Baseline Notebook

本 notebook 按以下顺序组织：
1. 读取两个原始数据集。
2. 分别输出内容信息与初筛结果。
3. 按时间尺度清洗并写入新的 DuckDB。
4. 在同一分词空间下构建 grant Description 与 arXiv 论文文本的 TF-IDF baseline。

当前时间窗口：
- grant.gov：2004 到当前快照中的 2026 数据
- arXiv：1995 起，到当前快照中的最新可用数据

In [1]:
import duckdb
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

from main import (
    ARXIV_START,
    GRANT_END,
    GRANT_NAMESPACE,
    GRANT_START,
    PROCESSED_DB_PATH,
    RAW_ARXIV_PATH,
    RAW_GRANT_PATH,
    build_database,
    clean_text,
    parse_grant_date,
)


In [2]:
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)

RAW_ARXIV_PATH, RAW_GRANT_PATH, PROCESSED_DB_PATH


(PosixPath('/Users/wanfang/repos/SRMatcher/data/raw/arxiv-metadata-oai-snapshot.json'),
 PosixPath('/Users/wanfang/repos/SRMatcher/data/raw/grant.gov.xml'),
 PosixPath('/Users/wanfang/repos/SRMatcher/data/processed/srmatcher.duckdb'))

## 1. Read Raw Data

In [3]:
grant_raw = pd.read_xml(
    RAW_GRANT_PATH,
    xpath=".//g:OpportunitySynopsisDetail_1_0",
    namespaces=GRANT_NAMESPACE,
    parser="lxml",
)

grant_raw[["OpportunityID", "OpportunityTitle", "AgencyName", "Description"]].head()


,OpportunityID,OpportunityTitle,AgencyName,Description
0,262148,Establishment of the Edmund S. Muskie Graduate Internship Program,Bureau of South and Central Asian Affairs,The Office of Press and Public Diplomacy of the Bureau of South and Central Asian Affairs of the U.S. Department of ...
1,262149,Eradication of Yellow Crazy Ants on Johnston Atoll NWR,Fish and Wildlife Service,Funds under this award are to be used for the eradication of Yellow Crazy Ants from Johnston Atoll National Wildlife...
2,131073,"Cooperative Ecosystem Studies Unit, Piedmont South Atlantic Coast CESU",Geological Survey,The USGS Southeast Ecological Science Center seeks to provide financial assistance for research investigating the us...
3,131094,"Plant Feedstock Genomics for Bioenergy: A Joint Research Funding Opportunity Announcement USDA, DOE",Office of Science,"The U.S. Department of Energy&apos;s Office of Science, Office of Biological and Environmental Research (BER), and t..."
4,131095,Management of HIV-Related Lung Disease and Cardiovascular Co-Morbidity (R34),National Institutes of Health,"This FOA invites clinical trials planning grant (R34) applications to support the initial organization, protocol dev..."


In [4]:
with duckdb.connect() as con:
    arxiv_raw_preview = con.execute(
        """
        SELECT
            id,
            title,
            categories,
            CAST(try_strptime(versions[1].created, '%a, %d %b %Y %H:%M:%S GMT') AS DATE) AS submitted_date,
            CAST(update_date AS DATE) AS updated_date,
            left(trim(regexp_replace(abstract, '\\s+', ' ', 'g')), 240) AS abstract_preview
        FROM read_json_auto(?, format = 'newline_delimited')
        LIMIT 5
        """,
        [str(RAW_ARXIV_PATH)],
    ).df()

arxiv_raw_preview


,id,title,categories,submitted_date,updated_date,abstract_preview
0,0704.0001,Calculation of prompt diphoton production cross sections at Tevatron and\n LHC energies,hep-ph,2007-04-02,2008-11-26,A fully differential calculation in perturbative quantum chromodynamics is presented for the production of massive p...
1,0704.0002,Sparsity-certifying Graph Decompositions,math.CO cs.CG,2007-03-31,2008-12-13,"We describe a new algorithm, the $(k,\ell)$-pebble game with colors, and use it obtain a characterization of the fam..."
2,0704.0003,The evolution of the Earth-Moon system based on the dark matter field\n fluid model,physics.gen-ph,2007-04-01,2008-01-13,The evolution of Earth-Moon system is described by the dark matter field fluid model proposed in the Meeting of Divi...
3,0704.0004,A determinant of Stirling cycle numbers counts unlabeled acyclic\n single-source automata,math.CO,2007-03-31,2007-05-23,We show that a determinant of Stirling cycle numbers counts unlabeled acyclic single-source automata. The proof invo...
4,0704.0005,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\alpha}$,math.CA math.FA,2007-04-02,2013-10-15,"In this paper we show how to compute the $\Lambda_{\alpha}$ norm, $\alpha\ge 0$, using the dyadic grid. This result ..."


## 2. Content Info and Initial Screening

In [5]:
grant_work = grant_raw.copy()

for column in ["PostDate", "CloseDate", "LastUpdatedDate", "ArchiveDate"]:
    if column in grant_work.columns:
        grant_work[f"{column}_parsed"] = parse_grant_date(grant_work[column])

grant_work["Description_clean"] = grant_work["Description"].map(clean_text)
grant_work["Eligibility_clean"] = grant_work["AdditionalInformationOnEligibility"].map(clean_text)

grant_info = pd.DataFrame(
    {
        "metric": [
            "raw_rows",
            "raw_columns",
            "min_posted_date",
            "max_posted_date",
            "missing_description",
            "duplicate_opportunity_id",
            "rows_in_target_window",
        ],
        "value": [
            len(grant_work),
            grant_work.shape[1],
            grant_work["PostDate_parsed"].min(),
            grant_work["PostDate_parsed"].max(),
            int(grant_work["Description_clean"].isna().sum()),
            int(grant_work["OpportunityID"].duplicated().sum()),
            int(grant_work["PostDate_parsed"].between(GRANT_START, GRANT_END).sum()),
        ],
    }
)

grant_screened = grant_work.loc[
    grant_work["PostDate_parsed"].between(GRANT_START, GRANT_END)
    & grant_work["Description_clean"].notna()
].copy()

grant_info


,metric,value
0,raw_rows,80763
1,raw_columns,36
2,min_posted_date,2004-03-22 00:00:00
3,max_posted_date,2026-03-11 00:00:00
4,missing_description,15
5,duplicate_opportunity_id,0
6,rows_in_target_window,80763


In [6]:
grant_screened[[
    "OpportunityID",
    "OpportunityTitle",
    "AgencyName",
    "PostDate_parsed",
    "Description_clean",
]].head()


,OpportunityID,OpportunityTitle,AgencyName,PostDate_parsed,Description_clean
0,262148,Establishment of the Edmund S. Muskie Graduate Internship Program,Bureau of South and Central Asian Affairs,2014-08-15,The Office of Press and Public Diplomacy of the Bureau of South and Central Asian Affairs of the U.S. Department of ...
1,262149,Eradication of Yellow Crazy Ants on Johnston Atoll NWR,Fish and Wildlife Service,2014-08-15,Funds under this award are to be used for the eradication of Yellow Crazy Ants from Johnston Atoll National Wildlife...
2,131073,"Cooperative Ecosystem Studies Unit, Piedmont South Atlantic Coast CESU",Geological Survey,2011-11-17,The USGS Southeast Ecological Science Center seeks to provide financial assistance for research investigating the us...
3,131094,"Plant Feedstock Genomics for Bioenergy: A Joint Research Funding Opportunity Announcement USDA, DOE",Office of Science,2011-11-17,"The U.S. Department of Energy's Office of Science, Office of Biological and Environmental Research (BER), and the U...."
4,131095,Management of HIV-Related Lung Disease and Cardiovascular Co-Morbidity (R34),National Institutes of Health,2011-11-17,"This FOA invites clinical trials planning grant (R34) applications to support the initial organization, protocol dev..."


In [7]:
with duckdb.connect() as con:
    arxiv_info = con.execute(
        """
        WITH raw AS (
            SELECT
                id,
                title,
                abstract,
                categories,
                CAST(try_strptime(versions[1].created, '%a, %d %b %Y %H:%M:%S GMT') AS DATE) AS submitted_date,
                CAST(update_date AS DATE) AS updated_date
            FROM read_json_auto(?, format = 'newline_delimited')
        )
        SELECT * FROM (
            SELECT 'raw_rows' AS metric, CAST(COUNT(*) AS VARCHAR) AS value FROM raw
            UNION ALL
            SELECT 'submitted_min', CAST(MIN(submitted_date) AS VARCHAR) FROM raw
            UNION ALL
            SELECT 'submitted_max', CAST(MAX(submitted_date) AS VARCHAR) FROM raw
            UNION ALL
            SELECT 'missing_abstract', CAST(SUM(CASE WHEN nullif(trim(abstract), '') IS NULL THEN 1 ELSE 0 END) AS VARCHAR) FROM raw
            UNION ALL
            SELECT 'rows_from_1995_forward', CAST(SUM(CASE WHEN submitted_date >= CAST(? AS DATE) THEN 1 ELSE 0 END) AS VARCHAR) FROM raw
        )
        """,
        [str(RAW_ARXIV_PATH), ARXIV_START],
    ).df()

arxiv_info


,metric,value
0,raw_rows,2975294
1,submitted_min,1986-04-25
2,submitted_max,2026-03-05
3,missing_abstract,0
4,rows_from_1995_forward,2954910


In [8]:
with duckdb.connect() as con:
    arxiv_screened_preview = con.execute(
        """
        SELECT
            id,
            title,
            categories,
            CAST(try_strptime(versions[1].created, '%a, %d %b %Y %H:%M:%S GMT') AS DATE) AS submitted_date,
            left(trim(regexp_replace(abstract, '\\s+', ' ', 'g')), 240) AS abstract_preview
        FROM read_json_auto(?, format = 'newline_delimited')
        WHERE CAST(try_strptime(versions[1].created, '%a, %d %b %Y %H:%M:%S GMT') AS DATE) >= CAST(? AS DATE)
          AND nullif(trim(abstract), '') IS NOT NULL
        ORDER BY submitted_date DESC
        LIMIT 5
        """,
        [str(RAW_ARXIV_PATH), ARXIV_START],
    ).df()

arxiv_screened_preview


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,id,title,categories,submitted_date,abstract_preview
0,2603.05248,Effective vertexes in magnetized quark-gluon plasma,hep-ph,2026-03-05,"In quark-gluon plasma (QGP), at high temperatures $T$ the spontaneous generation of color magnetic fields, $b^3(T), ..."
1,2603.05249,Robust and optimal control of open quantum systems,quant-ph,2026-03-05,"Recent advancements in quantum technologies have highlighted the importance of mitigating system imperfections, incl..."
2,2603.05250,A Benchmarking Framework for Model Datasets,cs.SE,2026-03-05,"Empirical and LLM-based research in model-driven engineering increasingly relies on datasets of software models, for..."
3,2603.05251,On Dual-Fed Pinching Antenna Systems with In-Waveguide Attenuation,eess.SP,2026-03-05,Pinching antenna systems (PAS) have recently emerged as a promising architecture for flexible and reconfigurable wir...
4,2603.05252,Rethinking the Role of Collaborative Robots in Rehabilitation,cs.RO,2026-03-05,Current research on collaborative robots (cobots) in physical rehabilitation largely focuses on repeated motion trai...


## 3. Time Alignment and Clean DuckDB Build

In [9]:
build_summary = build_database(replace=True)
build_summary


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{'database_path': '/Users/wanfang/repos/SRMatcher/data/processed/srmatcher.duckdb',
 'grant_clean': {'row_count': 80748,
  'min_date': datetime.datetime(2004, 3, 22, 0, 0),
  'max_date': datetime.datetime(2026, 3, 11, 0, 0)},
 'arxiv_clean': {'row_count': 2954910,
  'min_date': datetime.datetime(1995, 1, 1, 0, 0),
  'max_date': datetime.datetime(2026, 3, 5, 0, 0)}}

In [10]:
with duckdb.connect(PROCESSED_DB_PATH, read_only=True) as con:
    dataset_summary = con.execute(
        "SELECT * FROM dataset_summary ORDER BY dataset_name"
    ).df()

dataset_summary


,dataset_name,row_count,min_date,max_date
0,arxiv_clean,2954910,1995-01-01,2026-03-05
1,grant_clean,80748,2004-03-22,2026-03-11


In [11]:
with duckdb.connect(PROCESSED_DB_PATH, read_only=True) as con:
    grant_clean_preview = con.execute(
        """
        SELECT opportunity_id, opportunity_title, agency_name, posted_date, grant_year
        FROM grant_clean
        ORDER BY posted_date DESC, opportunity_id
        LIMIT 5
        """
    ).df()
    arxiv_clean_preview = con.execute(
        """
        SELECT arxiv_id, title, categories, submitted_date, submitted_year
        FROM arxiv_clean
        ORDER BY submitted_date DESC, arxiv_id
        LIMIT 5
        """
    ).df()

grant_clean_preview, arxiv_clean_preview


(  opportunity_id  \
 0         361441   
 1         361443   
 2         361444   
 3         361445   
 4         361446   
 
                                                                                                          opportunity_title  \
 0  Organic Agriculture Research and Extension Initiative - OREI Planning Projects for Assistance in Development of Futu...   
 1  Organic Agriculture Research and Extension Initiative - OREI Research Projects with Extension and/or Education Compo...   
 2                                           Organic Agriculture Research and Extension Initiative - OREI Workshop Projects   
 3                                            2026 DFC National Community Anti-Drug Institute Notice of Funding Opportunity   
 4                                                               Puget Sound Action Agenda - Strategic Implementation Leads   
 
                                agency_name posted_date  grant_year  
 0       Electronic Research Administr

## 4. TF-IDF Baseline in a Shared Token Space

说明：`grant_clean.demand_text` 与 `arxiv_clean.paper_text` 已经是同一文本空间中的输入文本。
其中论文侧用 `title + abstract + categories`，grant 侧用 `title + description + category + agency`。

全量两两匹配代价太高，这里先构造一个可运行的 baseline 子集，后续再扩到更大规模。

In [12]:
BASELINE_GRANT_LIMIT = 300
BASELINE_ARXIV_LIMIT = 30000
TOP_K = 5

with duckdb.connect(PROCESSED_DB_PATH, read_only=True) as con:
    grants_for_match = con.execute(
        """
        SELECT opportunity_id, opportunity_title, posted_date, demand_text
        FROM grant_clean
        WHERE demand_text IS NOT NULL
        ORDER BY posted_date DESC, opportunity_id
        LIMIT ?
        """,
        [BASELINE_GRANT_LIMIT],
    ).df()
    arxiv_for_match = con.execute(
        """
        SELECT arxiv_id, title, submitted_date, paper_text
        FROM arxiv_clean
        WHERE paper_text IS NOT NULL
        ORDER BY submitted_date DESC, arxiv_id
        LIMIT ?
        """,
        [BASELINE_ARXIV_LIMIT],
    ).df()

grants_for_match.shape, arxiv_for_match.shape


((300, 4), (30000, 4))

In [13]:
shared_corpus = pd.concat(
    [
        arxiv_for_match["paper_text"].fillna(""),
        grants_for_match["demand_text"].fillna(""),
    ],
    ignore_index=True,
)

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=50000,
    ngram_range=(1, 2),
    min_df=5,
    sublinear_tf=True,
)

tfidf_matrix = vectorizer.fit_transform(shared_corpus)
arxiv_matrix = tfidf_matrix[: len(arxiv_for_match)]
grant_matrix = tfidf_matrix[len(arxiv_for_match) :]

nn = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=TOP_K)
nn.fit(arxiv_matrix)
distances, indices = nn.kneighbors(grant_matrix)

tfidf_matrix.shape


(30300, 50000)

In [14]:
match_rows = []

for grant_idx, grant_row in grants_for_match.reset_index(drop=True).iterrows():
    for rank, (distance, paper_idx) in enumerate(zip(distances[grant_idx], indices[grant_idx]), start=1):
        paper_row = arxiv_for_match.iloc[paper_idx]
        match_rows.append(
            {
                "grant_rank": rank,
                "opportunity_id": grant_row["opportunity_id"],
                "opportunity_title": grant_row["opportunity_title"],
                "posted_date": grant_row["posted_date"],
                "arxiv_id": paper_row["arxiv_id"],
                "paper_title": paper_row["title"],
                "submitted_date": paper_row["submitted_date"],
                "similarity": 1 - float(distance),
            }
        )

baseline_matches = pd.DataFrame(match_rows).sort_values(
    ["opportunity_id", "grant_rank", "similarity"],
    ascending=[True, True, False],
)

baseline_matches.head(20)


,grant_rank,opportunity_id,opportunity_title,posted_date,arxiv_id,paper_title,submitted_date,similarity
1490,1,345316,2026 Tribal Transportation Program Safety Fund,2025-11-07,2602.24097,Bi-level RL-Heuristic Optimization for Real-world Winter Road Maintenance,2026-02-27,0.116551
1491,2,345316,2026 Tribal Transportation Program Safety Fund,2025-11-07,2602.20136,On a discrete max-plus transportation problem,2026-02-23,0.099559
1492,3,345316,2026 Tribal Transportation Program Safety Fund,2025-11-07,2602.24238,Time Series Foundation Models as Strong Baselines in Transportation Forecasting: A Large-Scale Benchmark Analysis,2026-02-27,0.097409
1493,4,345316,2026 Tribal Transportation Program Safety Fund,2025-11-07,2602.15554,Efficient Road Renovation Scheduling under Uncertainty using Lower Bound Pruning,2026-02-17,0.091692
1494,5,345316,2026 Tribal Transportation Program Safety Fund,2025-11-07,2602.12591,Vehicle behaviour estimation for abnormal event detection using distributed fiber optic sensing,2026-02-13,0.086928
1460,1,355578,FY 2025 Preschool Development Grant Birth Through Five (PDG B-5) Systems-Building Grant,2025-11-18,2602.10501,Division of Labor and Collaboration Between Parents in Family Education,2026-02-11,0.111105
1461,2,355578,FY 2025 Preschool Development Grant Birth Through Five (PDG B-5) Systems-Building Grant,2025-11-18,2602.12749,"SoK: Understanding the Pedagogical, Health, Ethical, and Privacy Challenges of Extended Reality in Early Childhood E...",2026-02-13,0.110501
1462,3,355578,FY 2025 Preschool Development Grant Birth Through Five (PDG B-5) Systems-Building Grant,2025-11-18,2602.10381,Deep learning outperforms traditional machine learning methods in predicting childhood malnutrition: evidence from s...,2026-02-11,0.106941
1463,4,355578,FY 2025 Preschool Development Grant Birth Through Five (PDG B-5) Systems-Building Grant,2025-11-18,2603.00996,Sustainable Care: Designing Technologies That Support Children's Long-Term Engagement with Social Issues,2026-03-01,0.106235
1464,5,355578,FY 2025 Preschool Development Grant Birth Through Five (PDG B-5) Systems-Building Grant,2025-11-18,2602.23095,TaleBot: A Tangible AI Companion to Support Children in Co-creative Storytelling for Resilience Cultivation,2026-02-26,0.100956


In [15]:
best_matches = baseline_matches.loc[baseline_matches["grant_rank"] == 1].sort_values(
    "similarity",
    ascending=False,
)

best_matches.head(20)


,grant_rank,opportunity_id,opportunity_title,posted_date,arxiv_id,paper_title,submitted_date,similarity
75,1,360976,Children&apos;s Mental Health Initiative,2026-03-06,2602.20378,The Role of Family Support in the Well-Being of Older People: Evidence from Malaysia and Viet Nam,2026-02-23,0.242995
225,1,361374,Rural Microentrepreneur Program(RMAP),2026-02-26,2602.09248,Reply To: Global Gridded Population Datasets Systematically Underrepresent Rural Population by Josias L\'ang-Ritter ...,2026-02-09,0.225679
295,1,361326,BJA FY25 Local Law Enforcement Crime Gun Intelligence Center Integration Initiative,2026-02-19,2603.02150,Zero- and Few-Shot Named-Entity Recognition: Case Study and Dataset in the Crime Domain (CrimeNER),2026-03-02,0.221172
845,1,361139,Laura Bush 21st Century Librarian Program (2026),2026-01-13,2602.17235,Weak 21st-century AMOC response to Greenland meltwater in a strongly eddying ocean model,2026-02-19,0.212534
1320,1,360923,Rare Earth Elements Demonstration Facility,2025-12-01,2603.03563,Absolute Primary Nanothermometry Using Individual Stark Sublevels of Rare-Earth-doped Crystals,2026-03-03,0.205765
290,1,361325,BJA FY25 Public Safety and Mental Health Initiative,2026-02-19,2602.08121,Initial Risk Probing and Feasibility Testing of Glow: a Generative AI-Powered Dialectical Behavior Therapy Skills Co...,2026-02-08,0.202573
865,1,361143,21st Century Museum Professionals Program (2026),2026-01-13,2602.17235,Weak 21st-century AMOC response to Greenland meltwater in a strongly eddying ocean model,2026-02-19,0.198799
1075,1,360555,Extension of the World Trade Center Health Registry (U50),2025-12-19,2602.03015,A Vision-Based Analysis of Congestion Pricing in New York City,2026-02-03,0.196997
825,1,361133,BJA FY25 Preventing Violence Against Law Enforcement Officers and Ensuring Officer Resilience and Survivability (VAL...,2026-01-13,2603.02150,Zero- and Few-Shot Named-Entity Recognition: Case Study and Dataset in the Crime Domain (CrimeNER),2026-03-02,0.196868
670,1,361190,"Logistical Support for INL Border Security and Counternarcotics, Criminal Deterrence and Transnational Crime and Cou...",2026-01-22,2602.16561,Hidden in Plain Sight: Detecting Illicit Massage Businesses from Mobility Data,2026-02-18,0.193138
